09_master_production_catalogue.ipynb

Cell 1:
Imports + setup

Cell 2:
Import/recreate get_season_productions()

Cell 3:
Investigate season discovery

Cell 4:
Build get_all_season_ids()

Cell 5:
Test season discovery

Cell 6:
Loop through seasons

Cell 7:
Build master dataframe

Cell 8:
Validation

Cell 9:
Export production_catalogue.csv

In [1]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import re
import time

In [2]:
def get_season_productions(season_id):
    """
    Extract all Broadway productions listed for a season.

    Parameters
    ----------
    season_id : int or str

    Returns
    -------
    pandas.DataFrame
        Columns:
        - production_id
        - title
        - url
        - season_id
    """

    url = f"https://www.ibdb.com/season/{season_id}"

    response = requests.get(url)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    rows = []

    for a in soup.find_all("a", href=True):
        href = a["href"]

        match = re.search(r"/broadway-production/.*-(\d+)$", href)

        if match:
            rows.append({
                "production_id": match.group(1),
                "title": a.get_text(strip=True),
                "url": "https://www.ibdb.com" + href,
                "season_id": season_id
            })

    df = (
        pd.DataFrame(rows)
        .drop_duplicates(subset="production_id")
        .reset_index(drop=True)
    )

    assert df["production_id"].is_unique
    assert df["production_id"].notna().all()

    return df

In [3]:
url = "https://www.ibdb.com/broadway-season"

response = requests.get(url)

print(response.status_code)
print(response.text[:500])

404




<!DOCTYPE html>
<html lang="en">
<head>

    <!-- Live Global site tag (gtag.js) - Google Analytics -->
<script async src="https://www.googletagmanager.com/gtag/js?id=UA-20752142-5"></script>
<script>
    window.dataLayer = window.dataLayer || [];
    function gtag() { dataLayer.push(arguments); }
    gtag('js', new Date());

    gtag('config', 'UA-20752142-5');
</script>

<!-- Global site tag (gtag.js) - Google Analytics (GA4) -->
<script async src="https://www.googletagm


In [4]:
soup = BeautifulSoup(response.text, "html.parser")

links = [
    a.get("href")
    for a in soup.find_all("a")
]

links[:20]

season_links = [
    link for link in links
    if link and "season" in link.lower()
]

season_links[:20]

['/season/']

In [38]:
def check_season_exists(season_id, retries=3):
    url = f"https://www.ibdb.com/season/{season_id}"
    
    for attempt in range(retries):
        try:
            response = requests.get(url, timeout=10)
            soup = BeautifulSoup(response.text, "html.parser")
            
            title = soup.find("title")
            
            if title is None:
                return False
            
            if title.get_text(strip=True) == "Error":
                return False
            
            return True
        
        except requests.exceptions.RequestException:
            time.sleep(2)
    
    print(f"Failed after retries: {season_id}")
    return False

In [41]:
valid_season_ids = []

for season_id in range(1000, 1400):
    if check_season_exists(season_id):
        valid_season_ids.append(season_id)
        print("Found:", season_id)

    time.sleep(1)

Found: 1001
Found: 1002
Found: 1003
Found: 1004
Found: 1005
Found: 1006
Found: 1007
Found: 1008
Found: 1009
Found: 1010
Found: 1011
Found: 1012
Found: 1013
Found: 1014
Found: 1015
Found: 1016
Found: 1017
Found: 1018
Found: 1019
Found: 1020
Found: 1021
Found: 1022
Found: 1023
Found: 1024
Found: 1025
Found: 1026
Found: 1027
Found: 1028
Found: 1029
Found: 1030
Found: 1031
Found: 1032
Found: 1033
Found: 1034
Found: 1035
Found: 1036
Found: 1037
Found: 1038
Found: 1039
Found: 1040
Found: 1041
Found: 1042
Found: 1043
Found: 1044
Found: 1045
Found: 1046
Found: 1047
Found: 1048
Found: 1049
Found: 1050
Found: 1051
Found: 1052
Found: 1053
Found: 1054
Found: 1055
Found: 1056
Found: 1057
Found: 1058
Found: 1059
Found: 1060
Found: 1061
Found: 1062
Found: 1063
Found: 1064
Found: 1065
Found: 1066
Found: 1067
Found: 1068
Found: 1069
Found: 1070
Found: 1071
Found: 1072
Found: 1073
Found: 1074
Found: 1075
Found: 1076
Found: 1077
Found: 1078
Found: 1079
Found: 1080
Found: 1081
Found: 1082
Found: 1083
Foun

In [42]:
len(valid_season_ids)

277

In [43]:
valid_season_ids[:10], valid_season_ids[-10:]

([1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010],
 [1287, 1288, 1289, 1290, 1291, 1295, 1296, 1297, 1298, 1299])

In [44]:
def get_season_label(season_id):
    url = f"https://www.ibdb.com/season/{season_id}"
    
    response = requests.get(url)
    soup = BeautifulSoup(response.text, "html.parser")
    
    text = soup.get_text(" ", strip=True)
    
    match = re.search(r"\d{4}-\d{4}", text)
    
    if match:
        return match.group()
    
    return None

In [51]:
import pandas as pd

pd.DataFrame({
    "season_id": valid_season_ids
}).to_csv(
    "valid_season_ids.csv",
    index=False
)

In [53]:
import time
import requests
from bs4 import BeautifulSoup
import re


def get_season_label(season_id, retries=5):
    url = f"https://www.ibdb.com/season/{season_id}"

    for attempt in range(retries):
        try:
            response = requests.get(
                url,
                timeout=15
            )

            soup = BeautifulSoup(response.text, "html.parser")

            text = soup.get_text(" ", strip=True)

            match = re.search(r"\d{4}-\d{4}", text)

            if match:
                return match.group()

            return None

        except requests.exceptions.RequestException as e:
            print(
                f"Retry {attempt+1}/{retries} failed for {season_id}"
            )
            time.sleep(3)

    print(f"Giving up on {season_id}")
    return None

    season_catalogue = []

for i, season_id in enumerate(valid_season_ids):

    print(f"{i+1}/{len(valid_season_ids)}: {season_id}")

    label = get_season_label(season_id)

    season_catalogue.append({
        "season_id": season_id,
        "season_label": label
    })

season_df = pd.DataFrame(season_catalogue)

season_df.head()

1/277: 1001
2/277: 1002
3/277: 1003
4/277: 1004
5/277: 1005
6/277: 1006
7/277: 1007
8/277: 1008
9/277: 1009
10/277: 1010
11/277: 1011
Retry 1/5 failed for 1011
12/277: 1012
13/277: 1013
14/277: 1014
15/277: 1015
16/277: 1016
17/277: 1017
18/277: 1018
19/277: 1019
20/277: 1020
21/277: 1021
22/277: 1022
23/277: 1023
24/277: 1024
25/277: 1025
26/277: 1026
27/277: 1027
28/277: 1028
29/277: 1029
30/277: 1030
31/277: 1031
32/277: 1032
33/277: 1033
34/277: 1034
35/277: 1035
36/277: 1036
37/277: 1037
38/277: 1038
39/277: 1039
40/277: 1040
41/277: 1041
42/277: 1042
43/277: 1043
44/277: 1044
45/277: 1045
46/277: 1046
47/277: 1047
48/277: 1048
49/277: 1049
50/277: 1050
51/277: 1051
52/277: 1052
53/277: 1053
54/277: 1054
55/277: 1055
56/277: 1056
57/277: 1057
58/277: 1058
59/277: 1059
60/277: 1060
61/277: 1061
62/277: 1062
63/277: 1063
64/277: 1064
65/277: 1065
66/277: 1066
67/277: 1067
68/277: 1068
69/277: 1069
70/277: 1070
71/277: 1071
72/277: 1072
73/277: 1073
74/277: 1074
75/277: 1075
76/277: 

,season_id,season_label
0,1001,1899-1900
1,1002,1900-1901
2,1003,1901-1902
3,1004,1902-1903
4,1005,1903-1904


In [58]:
season_catalogue = []

for i, season_id in enumerate(valid_season_ids):

    print(f"{i+1}/{len(valid_season_ids)}: {season_id}")

    label = get_season_label(season_id)

    season_catalogue.append({
        "season_id": season_id,
        "season_label": label
    })

season_df = pd.DataFrame(season_catalogue)

1/277: 1001
2/277: 1002
3/277: 1003
4/277: 1004
5/277: 1005
6/277: 1006
7/277: 1007
8/277: 1008
9/277: 1009
10/277: 1010
11/277: 1011
12/277: 1012
13/277: 1013
14/277: 1014
15/277: 1015
16/277: 1016
17/277: 1017
18/277: 1018
19/277: 1019
20/277: 1020
21/277: 1021
22/277: 1022
23/277: 1023
24/277: 1024
25/277: 1025
26/277: 1026
27/277: 1027
28/277: 1028
29/277: 1029
30/277: 1030
31/277: 1031
32/277: 1032
33/277: 1033
34/277: 1034
35/277: 1035
36/277: 1036
37/277: 1037
38/277: 1038
39/277: 1039
40/277: 1040
41/277: 1041
42/277: 1042
43/277: 1043
44/277: 1044
45/277: 1045
46/277: 1046
47/277: 1047
48/277: 1048
49/277: 1049
50/277: 1050
51/277: 1051
52/277: 1052
53/277: 1053
54/277: 1054
55/277: 1055
56/277: 1056
57/277: 1057
58/277: 1058
59/277: 1059
60/277: 1060
61/277: 1061
62/277: 1062
63/277: 1063
64/277: 1064
65/277: 1065
66/277: 1066
67/277: 1067
68/277: 1068
69/277: 1069
70/277: 1070
71/277: 1071
72/277: 1072
73/277: 1073
74/277: 1074
75/277: 1075
76/277: 1076
77/277: 1077
78/277: 

In [64]:
season_df.shape

(277, 2)

In [67]:
season_df.tail(20)

,season_id,season_label
257,1276,2007-2008
258,1277,2008-2009
259,1278,NaN
260,1280,NaN
261,1281,NaN
262,1282,NaN
263,1283,NaN
264,1284,NaN
265,1285,NaN
266,1286,NaN


In [68]:
season_df["season_label"].isna().sum()

np.int64(130)

In [69]:
season_df[season_df["season_label"].isna()]

,season_id,season_label
38,1039,NaN
39,1040,NaN
40,1041,NaN
41,1042,NaN
42,1043,NaN
...,...,...
267,1287,NaN
271,1291,NaN
272,1295,NaN
273,1296,NaN


In [75]:
season_catalogue = []
failed_seasons = []

for i, season_id in enumerate(valid_season_ids):

    print(f"{i+1}/{len(valid_season_ids)}: {season_id}")

    label = get_season_label(season_id)

    if label is None:
        failed_seasons.append(season_id)

    season_catalogue.append({
        "season_id": season_id,
        "season_label": label
    })

season_df = pd.DataFrame(season_catalogue)

print("\nFinished!")
print(f"Total seasons: {len(season_df)}")
print(f"Missing labels: {season_df['season_label'].isna().sum()}")
print(f"Failed season IDs: {failed_seasons}")

1/277: 1001
2/277: 1002
3/277: 1003
4/277: 1004
5/277: 1005
6/277: 1006
7/277: 1007
8/277: 1008
9/277: 1009
10/277: 1010
11/277: 1011
12/277: 1012
13/277: 1013
14/277: 1014
15/277: 1015
16/277: 1016
17/277: 1017
18/277: 1018
19/277: 1019
20/277: 1020
21/277: 1021
22/277: 1022
23/277: 1023
24/277: 1024
25/277: 1025
26/277: 1026
27/277: 1027
28/277: 1028
29/277: 1029
30/277: 1030
31/277: 1031
32/277: 1032
33/277: 1033
34/277: 1034
35/277: 1035
36/277: 1036
37/277: 1037
38/277: 1038
39/277: 1039
40/277: 1040
41/277: 1041
42/277: 1042
43/277: 1043
44/277: 1044
45/277: 1045
46/277: 1046
47/277: 1047
48/277: 1048
49/277: 1049
50/277: 1050
51/277: 1051
52/277: 1052
53/277: 1053
Retry 1/5 failed for 1053
54/277: 1054
55/277: 1055
56/277: 1056
57/277: 1057
58/277: 1058
59/277: 1059
60/277: 1060
61/277: 1061
62/277: 1062
63/277: 1063
64/277: 1064
65/277: 1065
66/277: 1066
67/277: 1067
68/277: 1068
69/277: 1069
70/277: 1070
71/277: 1071
72/277: 1072
73/277: 1073
74/277: 1074
75/277: 1075
76/277: 

In [77]:
retry_catalogue = []

for season_id in failed_seasons:
    print(season_id)
    
    label = get_season_label(season_id)

    retry_catalogue.append({
        "season_id": season_id,
        "season_label": label
    })

retry_df = pd.DataFrame(retry_catalogue)

print("Still missing:")
print(retry_df["season_label"].isna().sum())

retry_df

1039
1040
1041
1042
1043
1047
1048
1049
1050
1141
1142
1143
1144
1145
1146
1147
1148
1149
1150
1151
1152
1153
1154
1155
1156
1157
1158
1159
1160
1161
1162
1163
1164
1165
1166
1167
1168
1169
1219
1220
1221
1222
1223
1224
1225
1226
1227
1228
1229
1230
1231
1232
1233
1234
1238
1239
1240
1244
1245
1246
1247
1248
1270
1271
1272
1273
1277
1278
1280
1281
1282
1286
1287
1288
1289
1296
Retry 1/5 failed for 1296
1297
1298
1299
Still missing:
7


,season_id,season_label
0,1039,1937-1938
1,1040,1938-1939
2,1041,1939-1940
3,1042,1940-1941
4,1043,1941-1942
...,...,...
74,1289,NaN
75,1296,2023-2024
76,1297,2024-2025
77,1298,2025-2026


In [78]:
retry_df[retry_df["season_label"].isna()]

,season_id,season_label
68,1280,NaN
69,1281,NaN
70,1282,NaN
71,1286,NaN
72,1287,NaN
73,1288,NaN
74,1289,NaN


In [79]:
season_df = season_df.merge(
    retry_df,
    on="season_id",
    how="left",
    suffixes=("", "_retry")
)

In [86]:
season_df["season_label"] = (
    season_df["season_label"]
    .fillna(season_df["season_label_retry"])
)

In [87]:
season_df = season_df.drop(columns="season_label_retry")

In [88]:
season_df[season_df["season_label"].isna()]

,season_id,season_label
260,1280,NaN
261,1281,NaN
262,1282,NaN
266,1286,NaN
267,1287,NaN
268,1288,NaN
269,1289,NaN


In [91]:
season_df.to_csv(
    "season_catalogue.csv",
    index=False
)

In [92]:
season_df[season_df["season_label"].isna()]

,season_id,season_label
260,1280,NaN
261,1281,NaN
262,1282,NaN
266,1286,NaN
267,1287,NaN
268,1288,NaN
269,1289,NaN


In [93]:
season_df[
    season_df["season_id"].between(1275, 1290)
]

,season_id,season_label
256,1275,2006-2007
257,1276,2007-2008
258,1277,2008-2009
259,1278,2009-2010
260,1280,NaN
261,1281,NaN
262,1282,NaN
263,1283,2013-2014
264,1284,2014-2015
265,1285,2015-2016


In [94]:
season_overrides = pd.DataFrame({
    "season_id": [
        1280,
        1281,
        1282,
        1286,
        1287,
        1288,
        1289
    ],
    "season_label": [
        "2010-2011",
        "2011-2012",
        "2012-2013",
        "2016-2017",
        "2017-2018",
        "2018-2019",
        "2019-2020"
    ]
})

In [97]:
season_df = season_df.merge(
    season_overrides,
    on="season_id",
    how="left",
    suffixes=("", "_override")
)

season_df["season_label"] = (
    season_df["season_label"]
    .fillna(season_df["season_label_override"])
)

season_df = season_df.drop(columns="season_label_override")

In [98]:
season_df["season_label"].isna().sum()

np.int64(0)

In [106]:
season_df.to_csv(
    "season_catalogue.csv",
    index=False
)

In [103]:
season_df["start_year"] = (
    season_df["season_label"]
    .str.extract(r"(\d{4})")
    .astype(int)
)

In [105]:
season_df.tail()

,season_id,season_label,start_year
272,1295,2022-2023,2022
273,1296,2023-2024,2023
274,1297,2024-2025,2024
275,1298,2025-2026,2025
276,1299,2026-2027,2026


In [107]:
season_df["season_label"].duplicated().sum()

np.int64(0)